# 🧠 RNA GNN Energy-Based Model

Modèle GNN léger pour scorer et raffiner les structures RNA 3D.

**Architecture:**
- Input: Séquence + Coordonnées 3D
- Graph: Nucléotides = nœuds, Connexions = arêtes
- GNN: 2-3 couches de message passing
- Output: Score d'énergie (0-1)


In [ ]:
import numpy as np
import math
from pathlib import Path
from collections import Counter

# Vérifier si PyTorch est disponible
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
    TORCH_AVAILABLE = True
    print("✅ PyTorch disponible - Mode GNN activé")
except ImportError:
    TORCH_AVAILABLE = False
    print("⚠️ PyTorch non disponible - Mode fallback (règles physiques)")


## 1. Représentation en graphe


In [ ]:
# Encodage des nucléotides
NUCLEOTIDE_TO_IDX = {'A': 0, 'U': 1, 'G': 2, 'C': 3, 'N': 4}
IDX_TO_NUCLEOTIDE = {v: k for k, v in NUCLEOTIDE_TO_IDX.items()}
NUM_NUCLEOTIDES = 5

def sequence_to_onehot(sequence):
    """Convertit une séquence RNA en encodage one-hot."""
    n = len(sequence)
    onehot = np.zeros((n, NUM_NUCLEOTIDES), dtype=np.float32)
    for i, nuc in enumerate(sequence):
        idx = NUCLEOTIDE_TO_IDX.get(nuc, 4)  # 'N' pour inconnu
        onehot[i, idx] = 1.0
    return onehot

def coords_to_features(coords):
    """
    Extrait des features géométriques des coordonnées.
    - Position normalisée
    - Distance au centroïde
    - Direction locale
    """
    coords = np.array(coords, dtype=np.float32)
    n = len(coords)

    # Remplacer les coordonnées invalides (0,0,0)
    valid_mask = ~np.all(coords == 0, axis=1)
    if np.sum(valid_mask) > 0:
        centroid = np.mean(coords[valid_mask], axis=0)
    else:
        centroid = np.zeros(3)

    features = []
    for i in range(n):
        if not valid_mask[i]:
            # Coordonnées invalides
            features.append([0.0] * 9)
            continue

        # Position relative au centroïde
        rel_pos = coords[i] - centroid

        # Distance au centroïde
        dist_to_center = np.linalg.norm(rel_pos)

        # Direction locale (vecteur vers le prochain résidu)
        if i < n - 1 and valid_mask[i + 1]:
            direction = coords[i + 1] - coords[i]
            direction = direction / (np.linalg.norm(direction) + 1e-8)
        else:
            direction = np.zeros(3)

        # Normaliser la position
        rel_pos_norm = rel_pos / (np.linalg.norm(rel_pos) + 1e-8)

        feat = list(rel_pos_norm) + [dist_to_center / 50.0] + list(direction) + [i / n, (n - i) / n]
        features.append(feat)

    return np.array(features, dtype=np.float32)

def build_graph_edges(coords, k_neighbors=5, max_dist=15.0):
    """
    Construit les arêtes du graphe basées sur:
    - Connexions séquentielles (i, i+1)
    - K plus proches voisins
    """
    n = len(coords)
    coords = np.array(coords, dtype=np.float32)

    edges = []
    edge_features = []

    # Connexions séquentielles
    for i in range(n - 1):
        edges.append([i, i + 1])
        edges.append([i + 1, i])  # Bidirectionnel

        # Feature: distance + type (séquentiel=1)
        dist = np.linalg.norm(coords[i + 1] - coords[i])
        edge_features.append([dist / 10.0, 1.0, 0.0])
        edge_features.append([dist / 10.0, 1.0, 0.0])

    # K plus proches voisins (contacts spatiaux)
    valid_mask = ~np.all(coords == 0, axis=1)

    for i in range(n):
        if not valid_mask[i]:
            continue

        distances = []
        for j in range(n):
            if i == j or abs(i - j) <= 1 or not valid_mask[j]:
                distances.append(float('inf'))
            else:
                distances.append(np.linalg.norm(coords[i] - coords[j]))

        # Top-k voisins
        nearest = np.argsort(distances)[:k_neighbors]
        for j in nearest:
            if distances[j] < max_dist:
                edges.append([i, j])
                # Feature: distance + type (spatial=0, contact proche=1 si < 8Å)
                is_contact = 1.0 if distances[j] < 8.0 else 0.0
                edge_features.append([distances[j] / 10.0, 0.0, is_contact])

    if not edges:
        # Graphe vide fallback
        edges = [[0, 0]]
        edge_features = [[0.0, 0.0, 0.0]]

    return np.array(edges, dtype=np.int64), np.array(edge_features, dtype=np.float32)


## 2. Architecture GNN


In [ ]:
if TORCH_AVAILABLE:

    class GraphConvLayer(nn.Module):
        """
        Couche de convolution sur graphe simple.
        Message passing: h_i = σ(W1 * h_i + W2 * Σ(h_j * e_ij))
        """
        def __init__(self, in_dim, out_dim, edge_dim=3):
            super().__init__()
            self.linear_self = nn.Linear(in_dim, out_dim)
            self.linear_neighbor = nn.Linear(in_dim, out_dim)
            self.linear_edge = nn.Linear(edge_dim, out_dim)
            self.norm = nn.LayerNorm(out_dim)

        def forward(self, x, edge_index, edge_attr):
            """
            x: [N, in_dim] - node features
            edge_index: [E, 2] - edges (source, target)
            edge_attr: [E, edge_dim] - edge features
            """
            n_nodes = x.size(0)

            # Self transformation
            h_self = self.linear_self(x)

            # Aggregate neighbor messages
            h_neighbor = torch.zeros_like(h_self)

            for idx, (src, tgt) in enumerate(edge_index):
                src, tgt = int(src), int(tgt)
                # Message from src to tgt
                msg = self.linear_neighbor(x[src]) * torch.sigmoid(self.linear_edge(edge_attr[idx]))
                h_neighbor[tgt] += msg

            # Combine
            h = h_self + h_neighbor
            h = self.norm(h)
            h = F.relu(h)

            return h

    class RNAGraphNet(nn.Module):
        """
        GNN léger pour scorer les structures RNA.

        Architecture:
        - Input: sequence one-hot + coord features
        - 3 couches GraphConv
        - Global pooling (mean + max)
        - MLP pour prédire l'énergie
        """
        def __init__(self, node_in_dim=14, hidden_dim=64, num_layers=3, edge_dim=3):
            super().__init__()

            # Embedding initial
            self.input_proj = nn.Linear(node_in_dim, hidden_dim)

            # Couches GNN
            self.conv_layers = nn.ModuleList([
                GraphConvLayer(hidden_dim, hidden_dim, edge_dim)
                for _ in range(num_layers)
            ])

            # Sortie
            self.output_mlp = nn.Sequential(
                nn.Linear(hidden_dim * 2, hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(hidden_dim, 32),
                nn.ReLU(),
                nn.Linear(32, 1)
            )

        def forward(self, x, edge_index, edge_attr):
            """
            Retourne un score d'énergie (plus bas = meilleur).
            """
            # Projection initiale
            h = self.input_proj(x)

            # Message passing
            for conv in self.conv_layers:
                h_new = conv(h, edge_index, edge_attr)
                h = h + h_new  # Residual connection

            # Global pooling
            h_mean = h.mean(dim=0)
            h_max = h.max(dim=0)[0]
            h_global = torch.cat([h_mean, h_max])

            # Prédire l'énergie
            energy = self.output_mlp(h_global)

            return energy

    print("✅ Architecture GNN définie")


## 3. Dataset et Entraînement


In [ ]:
if TORCH_AVAILABLE:

    class RNAStructureDataset(Dataset):
        """
        Dataset pour entraîner le GNN EBM.
        Génère des paires (structure réelle, structure perturbée).
        """
        def __init__(self, structures, augment=True):
            """
            structures: liste de dicts {'sequence': str, 'coords': list}
            """
            self.structures = structures
            self.augment = augment

        def __len__(self):
            return len(self.structures) * 2  # Réel + perturbé

        def __getitem__(self, idx):
            struct_idx = idx // 2
            is_perturbed = idx % 2 == 1

            struct = self.structures[struct_idx]
            sequence = struct['sequence']
            coords = list(struct['coords'])

            if is_perturbed:
                # Perturber la structure
                coords = self._perturb_coords(coords)
                label = 1.0  # Haute énergie
            else:
                label = 0.0  # Basse énergie

            # Construire le graphe
            seq_onehot = sequence_to_onehot(sequence)
            coord_features = coords_to_features(coords)
            node_features = np.concatenate([seq_onehot, coord_features], axis=1)

            edges, edge_features = build_graph_edges(coords)

            return {
                'node_features': torch.tensor(node_features, dtype=torch.float32),
                'edge_index': torch.tensor(edges, dtype=torch.long),
                'edge_attr': torch.tensor(edge_features, dtype=torch.float32),
                'label': torch.tensor([label], dtype=torch.float32),
                'length': len(sequence)
            }

        def _perturb_coords(self, coords, noise_level=5.0):
            """Perturbe les coordonnées pour créer des exemples négatifs."""
            perturbed = []
            for c in coords:
                if c == (0.0, 0.0, 0.0) or c == [0.0, 0.0, 0.0]:
                    perturbed.append(c)
                else:
                    noise = np.random.normal(0, noise_level, 3)
                    perturbed.append((c[0] + noise[0], c[1] + noise[1], c[2] + noise[2]))
            return perturbed

    def train_gnn_ebm(model, train_structures, epochs=50, lr=0.001, device='cpu'):
        """
        Entraîne le GNN EBM avec apprentissage contrastif.
        """
        model = model.to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        dataset = RNAStructureDataset(train_structures)

        print(f"🏋️ Entraînement GNN EBM sur {len(train_structures)} structures...")

        model.train()
        for epoch in range(epochs):
            total_loss = 0
            n_batches = 0

            # Shuffle indices
            indices = list(range(len(dataset)))
            np.random.shuffle(indices)

            for idx in indices:
                sample = dataset[idx]

                # Forward
                node_feat = sample['node_features'].to(device)
                edge_idx = sample['edge_index'].to(device)
                edge_attr = sample['edge_attr'].to(device)
                label = sample['label'].to(device)

                energy = model(node_feat, edge_idx, edge_attr)

                # Loss: structures réelles = basse énergie, perturbées = haute énergie
                # Binary cross entropy sur sigmoid(energy)
                prob = torch.sigmoid(energy)
                loss = F.binary_cross_entropy(prob, label)

                # Backward
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                total_loss += loss.item()
                n_batches += 1

            if (epoch + 1) % 10 == 0:
                avg_loss = total_loss / n_batches
                print(f"   Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}")

        print("✅ Entraînement terminé")
        return model

    print("✅ Fonctions d'entraînement définies")


## 4. Scoring et Raffinement


In [ ]:
if TORCH_AVAILABLE:

    def score_structure_gnn(model, sequence, coords, device='cpu'):
        """
        Score une structure avec le GNN.
        Retourne un score entre 0 et 1 (plus haut = meilleur).
        """
        model.eval()

        with torch.no_grad():
            # Construire le graphe
            seq_onehot = sequence_to_onehot(sequence)
            coord_features = coords_to_features(coords)
            node_features = np.concatenate([seq_onehot, coord_features], axis=1)
            edges, edge_features = build_graph_edges(coords)

            # Tenseurs
            node_feat = torch.tensor(node_features, dtype=torch.float32).to(device)
            edge_idx = torch.tensor(edges, dtype=torch.long).to(device)
            edge_attr = torch.tensor(edge_features, dtype=torch.float32).to(device)

            # Prédire l'énergie
            energy = model(node_feat, edge_idx, edge_attr)

            # Convertir en score (1 - sigmoid pour que bas = bon)
            score = 1.0 - torch.sigmoid(energy).item()

        return score

    def refine_structure_gnn(model, sequence, coords, n_iterations=30, lr=0.5, device='cpu'):
        """
        Raffine une structure par descente de gradient sur l'énergie GNN.
        """
        model.eval()

        # Convertir en tenseur avec gradient
        coords_array = np.array(coords, dtype=np.float32)
        coords_tensor = torch.tensor(coords_array, requires_grad=True, device=device)

        optimizer = torch.optim.SGD([coords_tensor], lr=lr)

        # Pré-calculer les features de séquence (fixes)
        seq_onehot = torch.tensor(sequence_to_onehot(sequence), dtype=torch.float32).to(device)

        for iteration in range(n_iterations):
            optimizer.zero_grad()

            # Recalculer les features de coordonnées
            coords_np = coords_tensor.detach().cpu().numpy()
            coord_features = torch.tensor(coords_to_features(coords_np), dtype=torch.float32).to(device)
            node_features = torch.cat([seq_onehot, coord_features], dim=1)

            # Reconstruire le graphe
            edges, edge_features = build_graph_edges(coords_np)
            edge_idx = torch.tensor(edges, dtype=torch.long).to(device)
            edge_attr = torch.tensor(edge_features, dtype=torch.float32).to(device)

            # Forward avec gradient
            # Note: On ne peut pas vraiment backprop à travers la construction du graphe
            # Donc on utilise une approche de perturbation numérique
            energy = model(node_features, edge_idx, edge_attr)

            # Pour le gradient, on utilise une approximation
            # En pratique, on pourrait utiliser des techniques plus avancées
            if energy.requires_grad:
                energy.backward()
                optimizer.step()

        # Retourner les coordonnées raffinées
        refined = [tuple(c) for c in coords_tensor.detach().cpu().numpy()]
        return refined

    print("✅ Fonctions de scoring GNN définies")


## 5. Intégration avec le pipeline


In [ ]:
class GNNEnergyModel:
    """
    Wrapper pour utiliser le GNN EBM dans le pipeline de prédiction.
    Gère l'entraînement, le scoring et le raffinement.
    """

    def __init__(self, hidden_dim=64, num_layers=3, device=None):
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = None
        self.is_trained = False
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        if TORCH_AVAILABLE:
            # node_in_dim = 5 (one-hot) + 9 (coord features) = 14
            self.model = RNAGraphNet(
                node_in_dim=14,
                hidden_dim=hidden_dim,
                num_layers=num_layers,
                edge_dim=3
            )
            print(f"🧠 GNN EBM initialisé (device={self.device})")
        else:
            print("⚠️ PyTorch non disponible - GNN désactivé")

    def train(self, structures, epochs=50, lr=0.001):
        """
        Entraîne le modèle sur des structures connues.

        structures: liste de dicts {'sequence': str, 'coords': list}
        """
        if not TORCH_AVAILABLE or self.model is None:
            print("⚠️ Entraînement impossible sans PyTorch")
            return

        self.model = train_gnn_ebm(
            self.model, structures,
            epochs=epochs, lr=lr, device=self.device
        )
        self.is_trained = True

    def score(self, sequence, coords):
        """
        Score une structure (0-1, plus haut = meilleur).
        """
        if not TORCH_AVAILABLE or self.model is None or not self.is_trained:
            # Fallback aux règles physiques
            return self._score_physics(coords, len(sequence))

        return score_structure_gnn(self.model, sequence, coords, self.device)

    def refine(self, sequence, coords, n_iterations=20):
        """
        Raffine une structure.
        """
        if not TORCH_AVAILABLE or self.model is None or not self.is_trained:
            # Fallback au raffinement physique
            return self._refine_physics(coords)

        return refine_structure_gnn(
            self.model, sequence, coords,
            n_iterations=n_iterations, device=self.device
        )

    def rank_predictions(self, sequence, predictions):
        """
        Classe les prédictions par score (meilleur en premier).
        """
        scored = [(self.score(sequence, pred), pred) for pred in predictions]
        scored.sort(key=lambda x: x[0], reverse=True)
        return [pred for _, pred in scored]

    def _score_physics(self, coords, seq_length):
        """Score basé sur les règles physiques (fallback)."""
        energy = 0.0
        ideal_dist = 5.9

        for i in range(len(coords) - 1):
            if coords[i] == (0.0, 0.0, 0.0) or coords[i+1] == (0.0, 0.0, 0.0):
                continue
            dist = math.sqrt(sum((a-b)**2 for a, b in zip(coords[i], coords[i+1])))
            energy += (dist - ideal_dist)**2

        normalized = energy / seq_length if seq_length > 0 else energy
        return 1.0 / (1.0 + normalized)

    def _refine_physics(self, coords, n_iterations=20):
        """Raffinement basé sur les règles physiques (fallback)."""
        refined = [list(c) for c in coords]
        ideal_dist = 5.9

        for _ in range(n_iterations):
            for i in range(len(refined) - 1):
                if refined[i] == [0.0, 0.0, 0.0] or refined[i+1] == [0.0, 0.0, 0.0]:
                    continue

                dist = math.sqrt(sum((a-b)**2 for a, b in zip(refined[i], refined[i+1])))
                if dist < 0.1:
                    continue

                if abs(dist - ideal_dist) > 1.0:
                    direction = [(refined[i+1][d] - refined[i][d]) / dist for d in range(3)]
                    correction = (dist - ideal_dist) * 0.3
                    for d in range(3):
                        refined[i+1][d] -= direction[d] * correction

        return [tuple(c) for c in refined]

    def save(self, path):
        """Sauvegarde le modèle."""
        if TORCH_AVAILABLE and self.model is not None:
            torch.save({
                'model_state': self.model.state_dict(),
                'hidden_dim': self.hidden_dim,
                'num_layers': self.num_layers,
                'is_trained': self.is_trained
            }, path)
            print(f"💾 Modèle sauvegardé: {path}")

    def load(self, path):
        """Charge le modèle."""
        if TORCH_AVAILABLE:
            checkpoint = torch.load(path, map_location=self.device)
            self.hidden_dim = checkpoint['hidden_dim']
            self.num_layers = checkpoint['num_layers']
            self.model = RNAGraphNet(
                node_in_dim=14,
                hidden_dim=self.hidden_dim,
                num_layers=self.num_layers,
                edge_dim=3
            )
            self.model.load_state_dict(checkpoint['model_state'])
            self.model.to(self.device)
            self.is_trained = checkpoint['is_trained']
            print(f"📂 Modèle chargé: {path}")

print("✅ Classe GNNEnergyModel définie")


## 6. Exemple d'utilisation


In [ ]:
def demo_gnn_ebm():
    """Démonstration du GNN EBM."""

    print("=" * 60)
    print("           🧠 DEMO GNN ENERGY-BASED MODEL")
    print("=" * 60)

    # Créer des structures d'exemple
    example_structures = [
        {
            'sequence': 'GGGAAACCC',
            'coords': [(i * 5.9, np.sin(i) * 5, np.cos(i) * 5) for i in range(9)]
        },
        {
            'sequence': 'AUCGAUCGAU',
            'coords': [(i * 5.9, np.sin(i * 0.5) * 10, np.cos(i * 0.5) * 10) for i in range(10)]
        },
        {
            'sequence': 'GGGGCCCC',
            'coords': [(i * 5.9, i * 2, -i * 2) for i in range(8)]
        }
    ]

    # Initialiser le modèle
    gnn_ebm = GNNEnergyModel(hidden_dim=32, num_layers=2)

    if TORCH_AVAILABLE:
        # Entraîner sur les exemples
        print("\n🏋️ Entraînement sur structures d'exemple...")
        gnn_ebm.train(example_structures, epochs=30, lr=0.01)

        # Tester le scoring
        print("\n📊 Test de scoring:")
        for struct in example_structures:
            score = gnn_ebm.score(struct['sequence'], struct['coords'])
            print(f"   {struct['sequence'][:10]}... : score = {score:.4f}")

        # Tester sur une structure perturbée
        print("\n📊 Structure perturbée:")
        perturbed = [(c[0] + np.random.normal(0, 3), c[1] + np.random.normal(0, 3), c[2] + np.random.normal(0, 3))
                     for c in example_structures[0]['coords']]
        score_perturbed = gnn_ebm.score(example_structures[0]['sequence'], perturbed)
        print(f"   Score perturbé: {score_perturbed:.4f}")
    else:
        print("\n⚠️ Mode fallback (règles physiques)")
        for struct in example_structures:
            score = gnn_ebm.score(struct['sequence'], struct['coords'])
            print(f"   {struct['sequence'][:10]}... : score = {score:.4f}")

    print("\n" + "=" * 60)

    return gnn_ebm

# Exécuter la démo
if __name__ == "__main__":
    gnn_model = demo_gnn_ebm()


## 7. Intégration avec le notebook principal

Pour utiliser le GNN EBM dans le pipeline:

```python
# Dans rna_3d_folding_eda.py
from rna_gnn_ebm import GNNEnergyModel

# Initialiser
gnn_ebm = GNNEnergyModel(hidden_dim=64, num_layers=3)

# Entraîner sur les templates
train_structures = [{'sequence': t['sequence'], 'coords': t['coords']}
                    for t in templates_db[:500]]
gnn_ebm.train(train_structures, epochs=50)

# Utiliser pour scorer/raffiner
def generate_predictions_with_gnn(query_seq, templates_db, gnn_ebm):
    predictions = generate_diverse_predictions(query_seq, templates_db, use_ebm=False)

    # Raffiner avec GNN
    refined = [gnn_ebm.refine(query_seq, pred) for pred in predictions]

    # Classer par score GNN
    ranked = gnn_ebm.rank_predictions(query_seq, refined)

    return ranked
```


In [ ]:
print("""
📋 RÉSUMÉ DU MODULE GNN EBM:

1. ARCHITECTURE
   • GraphConvLayer: Message passing simple
   • RNAGraphNet: 3 couches GNN + pooling global
   • ~50K paramètres (très léger)

2. FEATURES
   • Nœuds: One-hot nucléotide + features géométriques
   • Arêtes: Connexions séquentielles + k-NN spatial

3. ENTRAÎNEMENT
   • Contrastif: structures réelles vs perturbées
   • Loss: Binary cross-entropy

4. UTILISATION
   • score(sequence, coords) → 0-1
   • refine(sequence, coords) → coords améliorées
   • rank_predictions(sequence, preds) → preds triées

5. FALLBACK
   • Sans PyTorch: règles physiques automatiques
""")
